# Droplet-coalescence analysis

Reads the silo output from this case directory and reports two resolution diagnostics:

1. **Cells per diameter (CPD)** — grid resolution across the droplet, per axis.
2. **Interface thickness** — width of the diffuse interface ($0.05 \le \alpha_1 \le 0.95$), in metres and in cells.

Works on both an IC-check run (`t_step_stop=1, t_step_save=1`) and a full run. The time-series cell is a no-op when only one step exists.

**Prerequisite:** `silo_hdf5/` must exist in this directory (run `./mfc.sh run case.py --case a` with `post_process` enabled, format=1).

**Python env:** must have `h5py`, `numpy`, `matplotlib`. The MFC toolchain venv works:

```
source ../../build/venv/bin/activate
jupyter lab analysis.ipynb
```

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MFC_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(MFC_ROOT / "toolchain"))

import numpy as np
import matplotlib.pyplot as plt
from mfc.viz import assemble_silo, discover_timesteps

D = 240e-6
R = D / 2.0

# Mirror of the CASES table in case.py. Keep in sync if case.py changes.
CASES = {
    "a": (0.2, 14.8, 0.20),  "b": (0.5, 23.6, 0.10),  "c": (8.6, 105.9, 0.08),
    "d": (15.2, 139.8, 0.08), "e": (19.4, 158.0, 0.05), "f": (32.8, 210.8, 0.08),
    "g": (37.2, 228.0, 0.01), "h": (61.4, 296.5, 0.06), "i": (61.3, 295.3, 0.11),
    "j": (56.3, 288.9, 0.13), "k": (70.8, 327.7, 0.25), "l": (48.1, 270.1, 0.39),
    "m": (60.1, 302.8, 0.55), "n": (65.1, 320.3, 0.49), "o": (60.8, 313.7, 0.68),
    "p": (64.9, 312.8, 0.71), "q": (48.8, 260.3, 0.72), "r": (14.5, 149.1, 0.34),
}

CASE_LABEL = "a"
_, _, B = CASES[CASE_LABEL]
sep = 0.55 * D
b_impact = B * D

DROPLETS = {
    "left":  (-sep, -b_impact / 2.0, 0.0),
    "right": (+sep, +b_impact / 2.0, 0.0),
}

CASE_DIR = str(NOTEBOOK_DIR)
STEP = None

In [ ]:
steps = discover_timesteps(CASE_DIR, "silo")
if not steps:
    raise RuntimeError(
        f"No silo timesteps found under {CASE_DIR!r}. "
        "Run pre_process + post_process (format=1) first."
    )

step = STEP if STEP is not None else steps[0]
if step not in steps:
    raise ValueError(f"step={step} not in available steps {steps}")

print(f"Available steps ({len(steps)}): {steps[:10]}{' ...' if len(steps) > 10 else ''}")
print(f"Using step: {step}")

In [ ]:
data = assemble_silo(CASE_DIR, step)
print(f"ndim = {data.ndim}")
print(f"grid = {len(data.x_cc)} x {len(data.y_cc)} x {len(data.z_cc)}")
print(f"variables = {sorted(data.variables)}")

ALPHA_NAME = next((n for n in ("alpha1", "alpha_1") if n in data.variables), None)
if ALPHA_NAME is None:
    raise KeyError(f"No alpha1-like variable in {sorted(data.variables)}")
alpha = data.variables[ALPHA_NAME]
print(f"alpha: {ALPHA_NAME}  shape={alpha.shape}  range=[{alpha.min():.3g}, {alpha.max():.3g}]")

In [ ]:
x, y, z = data.x_cc, data.y_cc, data.z_cc

def _axis_stats(coord):
    d = np.diff(coord)
    return d.mean(), d.min(), d.max()

stats = {
    "x": (len(x), *_axis_stats(x)),
    "y": (len(y), *_axis_stats(y)),
    "z": (len(z), *_axis_stats(z)),
}

print(f"{'axis':<5}{'N':>6}{'Δ [um]':>12}{'CPD = D/Δ':>14}{'nonuniformity':>18}")
print("-" * 55)
for name, (n, d, dmin, dmax) in stats.items():
    u = (dmax - dmin) / max(dmin, 1e-300)
    tag = "uniform" if u < 1e-6 else f"{u:.1e}"
    print(f"{name:<5}{n:>6d}{d*1e6:>12.4f}{D/d:>14.2f}{tag:>18}")

dx = stats["x"][1]
dy = stats["y"][1]
dz = stats["z"][1]

In [ ]:
def interface_thickness_x_ray(alpha, x_cc, y_cc, z_cc, xc, yc, zc, R):
    jy = int(np.argmin(np.abs(y_cc - yc)))
    kz = int(np.argmin(np.abs(z_cc - zc)))

    if xc >= 0:
        mask = (x_cc >= xc) & (x_cc <= xc + 2.0 * R)
    else:
        mask = (x_cc <= xc) & (x_cc >= xc - 2.0 * R)

    x_seg = x_cc[mask]
    a_seg = alpha[mask, jy, kz]

    if a_seg.min() > 0.05 or a_seg.max() < 0.95:
        raise RuntimeError(
            f"alpha range on ray = [{a_seg.min():.3g}, {a_seg.max():.3g}]; "
            "interface not fully captured on this ray"
        )

    order = np.argsort(a_seg)
    a_sorted, x_sorted = a_seg[order], x_seg[order]
    x_05 = float(np.interp(0.05, a_sorted, x_sorted))
    x_95 = float(np.interp(0.95, a_sorted, x_sorted))

    t_m = abs(x_95 - x_05)
    dx_local = float(np.mean(np.diff(x_seg)))
    return t_m, t_m / dx_local, x_seg, a_seg


fig, ax = plt.subplots(figsize=(7, 4))
results = {}
for label, (xc, yc, zc) in DROPLETS.items():
    t_m, t_c, x_seg, a_seg = interface_thickness_x_ray(alpha, x, y, z, xc, yc, zc, R)
    results[label] = (t_m, t_c)
    ax.plot((x_seg - xc) / D, a_seg, "o-", ms=3, label=f"{label}: {t_c:.2f} cells ({t_m*1e6:.2f} um)")

ax.axhline(0.05, color="k", lw=0.5, ls=":")
ax.axhline(0.95, color="k", lw=0.5, ls=":")
ax.set_xlabel(r"$(x - x_c) / D$")
ax.set_ylabel(r"$\alpha_1$")
ax.set_title(f"Interface profile along x (step {step})")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

t_mean_cells = float(np.mean([v[1] for v in results.values()]))
print(f"Interface thickness (mean over droplets): {t_mean_cells:.2f} cells")
for k, (t_m, t_c) in results.items():
    print(f"  {k:>5}: {t_c:.2f} cells = {t_m*1e6:.2f} um")

In [ ]:
for label, (xc, yc, zc) in DROPLETS.items():
    mx = (x >= xc - 1.5 * R) & (x <= xc + 1.5 * R)
    my = (y >= yc - 1.5 * R) & (y <= yc + 1.5 * R)
    mz = (z >= zc - 1.5 * R) & (z <= zc + 1.5 * R)
    a_box = alpha[np.ix_(mx, my, mz)]
    gx_box = np.gradient(a_box, x[mx], axis=0)
    gy_box = np.gradient(a_box, y[my], axis=1)
    gz_box = np.gradient(a_box, z[mz], axis=2)
    gmag = np.sqrt(gx_box**2 + gy_box**2 + gz_box**2)
    t_grad_m = 0.9 / gmag.max()
    dx_local = float(np.mean(np.diff(x[mx])))
    print(f"{label:>5}: max|∇α|={gmag.max():.3g} 1/m  →  thickness ≈ {t_grad_m*1e6:.2f} um ({t_grad_m/dx_local:.2f} cells)")

In [ ]:
kz0 = int(np.argmin(np.abs(z - 0.0)))
slice_xy = alpha[:, :, kz0]

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(
    slice_xy.T,
    origin="lower",
    extent=[x[0] / D, x[-1] / D, y[0] / D, y[-1] / D],
    cmap="viridis",
    aspect="equal",
    vmin=0.0,
    vmax=1.0,
)
ax.contour(
    x / D, y / D, slice_xy.T,
    levels=[0.05, 0.5, 0.95],
    colors=["w", "r", "w"],
    linewidths=[0.6, 1.0, 0.6],
)
for label, (xc, yc, _zc) in DROPLETS.items():
    ax.plot(xc / D, yc / D, "r+", ms=12, mew=2)
ax.set_xlabel("x / D")
ax.set_ylabel("y / D")
ax.set_title(f"alpha_1 on z=0 midplane (step {step})")
fig.colorbar(im, ax=ax, label=r"$\alpha_1$")
plt.show()

In [ ]:
if len(steps) < 2:
    print("Only one step available — skipping time-series.")
else:
    series = {k: [] for k in DROPLETS}
    for s in steps:
        d_s = assemble_silo(CASE_DIR, s)
        a_s = d_s.variables[ALPHA_NAME]
        xs, ys, zs = d_s.x_cc, d_s.y_cc, d_s.z_cc
        for label, (xc0, _yc0, _zc0) in DROPLETS.items():
            mx = (xs >= xc0 - 1.5 * R) & (xs <= xc0 + 1.5 * R)
            w = a_s[mx, :, :]
            mass = float(w.sum())
            if mass <= 0:
                series[label].append(np.nan)
                continue
            xw = float(np.sum(w.sum(axis=(1, 2)) * xs[mx]) / mass)
            yw = float(np.sum(w.sum(axis=(0, 2)) * ys) / mass)
            zw = float(np.sum(w.sum(axis=(0, 1)) * zs) / mass)
            try:
                _, t_c, _, _ = interface_thickness_x_ray(a_s, xs, ys, zs, xw, yw, zw, R)
            except RuntimeError:
                t_c = np.nan
            series[label].append(t_c)

    fig, ax = plt.subplots(figsize=(7, 4))
    for label, vals in series.items():
        ax.plot(steps, vals, "o-", label=label)
    ax.set_xlabel("step")
    ax.set_ylabel("interface thickness [cells]")
    ax.set_title("Interface thickness over time")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

In [ ]:
print(f"case: {CASE_LABEL}   step: {step}")
print(f"grid: {stats['x'][0]} x {stats['y'][0]} x {stats['z'][0]}")
print(f"CPD : x={D/dx:.1f}  y={D/dy:.1f}  z={D/dz:.1f}")
for k, (t_m, t_c) in results.items():
    print(f"interface thickness ({k:>5}): {t_c:.2f} cells = {t_m*1e6:.2f} um")
print(f"mean interface thickness: {t_mean_cells:.2f} cells")